## Cell 1: Environment Setup & Kaggle Dataset Download


In [1]:
!nvidia-smi

Sun Nov 15 20:33:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q torchmetrics albumentations ultralytics kaggle

import json, os, subprocess
from pathlib import Path
from getpass import getpass

# 1. Configure Kaggle credentials
raw = getpass("Paste your Kaggle token or kaggle.json content: ").strip()
if raw.startswith("export KAGGLE_API_TOKEN="):
    raw = raw.split("=", 1)[1].strip().strip('"').strip("'")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

if raw.startswith("{"):
    creds = json.loads(raw)
    os.environ["KAGGLE_USERNAME"] = creds["username"]
    os.environ["KAGGLE_KEY"] = creds["key"]
    p = kaggle_dir / "kaggle.json"
    p.write_text(json.dumps(creds))
    p.chmod(0o600)
elif raw.startswith("KGAT_"):
    os.environ["KAGGLE_API_TOKEN"] = raw
    p = kaggle_dir / "access_token"
    p.write_text(raw)
    p.chmod(0o600)

# 2. Download and unzip dataset into /content/data if not already present
data_dir = Path("/content/data")
if not (data_dir / "train").is_dir():
    print("Downloading dataset from Kaggle...")
    !kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026 -p /content
    !unzip -oq /content/multiple-object-video-detection-hw-2-ucu-ai-2026.zip -d /content/data
    !rm -f /content/multiple-object-video-detection-hw-2-ucu-ai-2026.zip
    print("Dataset extracted successfully to /content/data")

import cv2
import logging
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision
import torchmetrics
from ultralytics import RTDETR, YOLO

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
LOGGER = logging.getLogger("visdrone_best_pipeline")
if torch.cuda.is_available():
    DEVICE = 0
    LOGGER.info("GPU=%s | PyTorch=%s | CUDA=%s", torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    LOGGER.info("GPU=MPS | PyTorch=%s", torch.__version__)
else:
    DEVICE = "cpu"
    LOGGER.warning("No GPU available; using CPU.")


100% 9.88G/9.88G [09:00<00:00, 19.6MB/s]  

Dataset extracted successfully to /content/data
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## Cell 2: Shared Configuration & High-Resolution RT-DETR Settings


In [3]:
from pathlib import Path
import logging
import torch

BASE_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
DATA_ROOT = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
LOCAL_OUTPUT_DIR = OUTPUT_DIR
OUTPUT_LOG_DIR = OUTPUT_DIR / "logs"
CHECKPOINT_OUTPUT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR = CHECKPOINT_OUTPUT_DIR
EDA_OUTPUT_DIR = OUTPUT_DIR / "eda"
FAILURE_OUTPUT_DIR = OUTPUT_DIR / "failures"
SUBMISSION_DIR = OUTPUT_DIR / "submissions"
for directory in (OUTPUT_DIR, OUTPUT_LOG_DIR, CHECKPOINT_DIR, EDA_OUTPUT_DIR, FAILURE_OUTPUT_DIR, SUBMISSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SHARED_CONFIG = {
    "manifest": str(EDA_OUTPUT_DIR / "sequence_split.csv"),
    "frame_skip": 5,
    "image_size": 800,
    "batch_size": 8 if torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0) else 16,
    "epochs": 12,
    "seed": 42,
    "mask_ignore_pixels": True,
    "max_detections": 500,
    "confidence_threshold": 0.20,
    "num_classes": 8,
    "class_names": [
        "pedestrian", "people", "bicycle", "car",
        "van", "truck", "tricycle", "bus",
    ],
    "save_dir": str(CHECKPOINT_OUTPUT_DIR),
}

TRAIN_CONFIG = {
    **SHARED_CONFIG,
    "device": 0 if torch.cuda.is_available() else DEVICE,
    "batch": SHARED_CONFIG["batch_size"],
}
RTDETR_EXPERIMENT_CONFIG = {
    "model": "rtdetr-l.pt",
    "imgsz": 800,
    "num_queries": 500,
    "matching_cost_class": 2.0,
    "matching_cost_bbox": 5.0,
    "matching_cost_giou": 2.0,
}
LOGGER.info("Resolved data root=%s | output=%s | batch=%d", DATA_ROOT, OUTPUT_DIR, TRAIN_CONFIG["batch"])


## Cell 3: Manifest Loading & VisDrone Dataset Preparation


In [4]:
!rm -f /content/drive/MyDrive/sequence_split.csv /content/outputs/eda/sequence_split.csv /content/data/sequence_split.csv

import csv
import math
import re
from collections import defaultdict
from dataclasses import dataclass
from typing import Any, Sequence

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch import Tensor
from torch.utils.data import Dataset

CLASS_NAMES = SHARED_CONFIG["class_names"]
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)
TRAILING_NUMBER = re.compile(r"-\d+$")
SPLIT_ROOTS: dict[str, Path] = {}


@dataclass(frozen=True)
class ImageRecord:
    image_path: Path
    source_split: str
    assigned_split: str
    clip_id: str
    frame_id: str


def normalized_stem(stem: str) -> str:
    return TRAILING_NUMBER.sub("", stem)


def _resolve_split_root(data_root: Path, split_name: str) -> Path | None:
    split_token = split_name.lower()
    search_roots = [data_root]
    for optional_root in (Path("/content"), Path("/content/drive/MyDrive")):
        if optional_root.is_dir() and optional_root not in search_roots:
            search_roots.append(optional_root)

    for search_root in search_roots:
        candidates = [
            search_root / split_name,
            search_root / f"VisDrone2019-DET-{split_name}",
            search_root / f"VisDrone2019-DET-{split_name}-dev",
            search_root / f"{split_name}_images",
        ]
        if search_root.is_dir():
            candidates.extend(
                child for child in search_root.iterdir()
                if child.is_dir() and split_token in child.name.lower()
            )
        for candidate in candidates:
            if not candidate.is_dir():
                continue
            image_root = _image_root(candidate)
            if image_root.is_dir() and next(
                (item for item in image_root.rglob("*")
                 if item.is_file() and item.suffix.lower() in IMAGE_SUFFIXES),
                None,
            ) is not None:
                SPLIT_ROOTS[split_name] = candidate.resolve()
                return SPLIT_ROOTS[split_name]
    return None


def resolve_manifest_image(
    raw_path: str,
    source_split: str,
    data_root: Path,
    manifest_path: Path,
) -> Path | None:
    recorded_path = Path(raw_path)
    
    # 1. Look for image filename directly under data_root / source_split
    candidates = [
        data_root / source_split / recorded_path.name,
        data_root / source_split / "images" / recorded_path.name,
        data_root / recorded_path.name,
    ]
    
    # 2. If recorded_path has clip subfolders, preserve the clip subfolder relative path
    if len(recorded_path.parts) >= 2:
        clip_name = recorded_path.parts[-2]
        candidates.append(data_root / source_split / clip_name / recorded_path.name)
        candidates.append(data_root / source_split / "images" / clip_name / recorded_path.name)
        
    for candidate in candidates:
        if candidate.is_file() and candidate.suffix.lower() in IMAGE_SUFFIXES:
            return candidate.resolve()
            
    # 3. Recursive fallback: find recorded_path.name anywhere inside data_root
    if data_root.is_dir():
        found = next((p for p in data_root.rglob(recorded_path.name) if p.is_file()), None)
        if found is not None:
            return found.resolve()
            
    return None


def load_manifest_records(manifest_path: Path, data_root: Path) -> list[ImageRecord]:
    required_columns = {"clip_id", "assigned_split", "source_split", "frame_id", "image_path"}
    records: list[ImageRecord] = []
    with manifest_path.open(newline="", encoding="utf-8") as manifest:
        reader = csv.DictReader(manifest)
        if not reader.fieldnames or not required_columns.issubset(reader.fieldnames):
            raise ValueError(f"manifest must contain columns {', '.join(sorted(required_columns))}")
        for line_number, row in enumerate(reader, start=2):
            source_split = (row.get("source_split") or "").strip()
            assigned_split = (row.get("assigned_split") or "").strip().lower()
            clip_id = (row.get("clip_id") or "").strip()
            frame_id = (row.get("frame_id") or "").strip()
            raw_path = (row.get("image_path") or "").strip()
            if source_split not in {"train", "val"} or assigned_split not in {"train", "val"} or not clip_id or not raw_path:
                raise ValueError(f"{manifest_path}:{line_number}: invalid split manifest row")
            image_path = resolve_manifest_image(raw_path, source_split, data_root, manifest_path)
            if image_path is None:
                raise ValueError(f"{manifest_path}:{line_number}: cannot resolve image {raw_path!r} under {data_root}")
            records.append(ImageRecord(image_path, source_split, assigned_split, clip_id, frame_id or image_path.stem))

    if not records:
        raise ValueError(f"manifest contains no image rows: {manifest_path}")
    if not any(record.assigned_split == "train" for record in records):
        raise ValueError("manifest has no training images")
    if not any(record.assigned_split == "val" for record in records):
        raise ValueError("manifest has no validation images")
    clip_splits: dict[str, set[str]] = defaultdict(set)
    for record in records:
        clip_splits[record.clip_id].add(record.assigned_split)
    leaking_clips = [clip for clip, splits in clip_splits.items() if len(splits) > 1]
    if leaking_clips:
        raise ValueError("manifest leaks clips across train/val: " + ", ".join(sorted(leaking_clips)))
    return records


def _image_root(split_root: Path) -> Path:
    candidate = split_root / "images"
    return candidate if candidate.is_dir() else split_root


def _scan_split(data_root: Path, split_name: str) -> list[ImageRecord]:
    split_root = SPLIT_ROOTS.get(split_name) or _resolve_split_root(data_root, split_name)
    if split_root is None:
        raise FileNotFoundError(
            f"Could not find a {split_name!r} image folder under {data_root}, /content, "
            "or /content/drive/MyDrive. Expected train/val or VisDrone2019-DET-train/val."
        )
    image_root = _image_root(split_root)
    image_paths = sorted(
        path for path in image_root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if not image_paths:
        raise FileNotFoundError(f"no image files found under {image_root}")

    records: list[ImageRecord] = []
    for image_path in image_paths:
        relative = image_path.relative_to(image_root)
        clip_id = relative.parts[0] if len(relative.parts) > 1 else f"flat:{split_name}"
        records.append(ImageRecord(image_path.resolve(), split_name, split_name, clip_id, image_path.stem))
    return records


def load_records(manifest: Path, data_root: Path) -> tuple[list[ImageRecord], str]:
    if manifest.is_file():
        records = load_manifest_records(manifest, data_root)
        return records, f"manifest: {manifest}"
    print(f"WARNING: manifest not found at {manifest}; scanning split folders.")
    records = _scan_split(data_root, "train") + _scan_split(data_root, "val")
    return records, f"scanned: {data_root / 'train'} and {data_root / 'val'}"


def clip_box(box: Sequence[float], image_width: int, image_height: int) -> list[float] | None:
    x1, y1, x2, y2 = box
    clipped = [
        min(max(float(x1), 0.0), float(image_width)),
        min(max(float(y1), 0.0), float(image_height)),
        min(max(float(x2), 0.0), float(image_width)),
        min(max(float(y2), 0.0), float(image_height)),
    ]
    if clipped[2] <= clipped[0] or clipped[3] <= clipped[1]:
        return None
    return clipped


def _annotation_boxes(
    annotation_path: Path,
    image_width: int,
    image_height: int,
) -> tuple[list[list[float]], list[int], list[list[float]]]:
    object_boxes: list[list[float]] = []
    object_labels: list[int] = []
    ignore_boxes: list[list[float]] = []
    try:
        rows = annotation_path.read_text(encoding="utf-8").splitlines()
    except OSError as error:
        raise OSError(f"could not read annotation {annotation_path}: {error}") from error

    for line_number, row in enumerate(rows, start=1):
        fields = row.split()
        if not fields:
            continue
        if len(fields) not in {4, 5}:
            raise ValueError(f"{annotation_path}:{line_number}: expected 4 ignore or 5 object values")
        try:
            values = [float(field) for field in fields]
        except ValueError as error:
            raise ValueError(f"{annotation_path}:{line_number}: annotation values must be numeric") from error
        if not all(math.isfinite(value) for value in values):
            raise ValueError(f"{annotation_path}:{line_number}: values must be finite")

        if len(values) == 5:
            raw_class, cx, cy, width, height = values
            class_id = int(raw_class)
            if raw_class != class_id or not 0 <= class_id < len(CLASS_NAMES):
                raise ValueError(f"{annotation_path}:{line_number}: class id must be in [0, 7]")
        else:
            cx, cy, width, height = values
            class_id = -1
        if width <= 0 or height <= 0:
            raise ValueError(f"{annotation_path}:{line_number}: box width and height must be positive")

        box = clip_box(
            ((cx - width / 2) * image_width, (cy - height / 2) * image_height,
             (cx + width / 2) * image_width, (cy + height / 2) * image_height),
            image_width, image_height,
        )
        if box is None:
            continue
        if class_id >= 0:
            object_boxes.append(box)
            object_labels.append(class_id + 1)
        else:
            ignore_boxes.append(box)
    return object_boxes, object_labels, ignore_boxes


def _flat_annotation_paths(split_root: Path) -> dict[str, list[Path]]:
    index: dict[str, list[Path]] = defaultdict(list)
    for path in sorted(split_root.rglob("*.txt")):
        if "labels" not in path.parts and "ignore" not in path.parts:
            index[normalized_stem(path.stem)].append(path)
    return index


def _find_annotation_paths(
    record: ImageRecord,
    split_root: Path,
    flat_index: dict[str, list[Path]],
) -> list[Path]:
    split_root = split_root.resolve()
    image_root = _image_root(split_root)
    try:
        relative_image = record.image_path.resolve().relative_to(image_root)
    except ValueError:
        relative_image = Path(record.image_path.name)

    labels_root, ignore_root = split_root / "labels", split_root / "ignore"
    if labels_root.is_dir() or ignore_root.is_dir():
        return [
            root / relative_image.with_suffix(".txt")
            for root in (labels_root, ignore_root)
            if (root / relative_image.with_suffix(".txt")).is_file()
        ]

    candidates = flat_index.get(normalized_stem(record.image_path.stem), [])
    selected: set[Path] = set()
    for expected_values in (4, 5):
        matching, empty = [], []
        for path in candidates:
            widths = {
                len(row.split()) for row in path.read_text(encoding="utf-8").splitlines()
                if row.split()
            }
            if expected_values in widths:
                matching.append(path)
            elif not widths:
                empty.append(path)
        exact = [path for path in matching if path.stem == record.image_path.stem]
        if exact:
            selected.update(exact)
        elif matching:
            selected.update(matching)
        elif expected_values == 5:
            selected.update(path for path in empty if path.stem == record.image_path.stem)
    return sorted(selected)


class VisDroneDetectionDataset(Dataset):
    """Same resize, normalization, ignore handling, and label mapping as hw_2.ipynb."""

    def __init__(
        self,
        records: Sequence[ImageRecord],
        data_root: Path,
        split: str,
        image_size: int = 448,
        mask_ignore_pixels: bool = True,
    ) -> None:
        self.records = [record for record in records if record.assigned_split == split]
        if not self.records:
            raise ValueError(f"no {split} records were provided")
        self.data_root = data_root
        self.split = split
        self.image_size = image_size
        self.mask_ignore_pixels = mask_ignore_pixels
        self._split_roots = {
            source: (SPLIT_ROOTS.get(source) or (data_root / source)).resolve()
            for source in {record.source_split for record in self.records}
        }
        self._flat_indexes = {
            source: _flat_annotation_paths(root)
            for source, root in self._split_roots.items()
            if not (root / "labels").is_dir() and not (root / "ignore").is_dir()
        }
        transforms: list[Any] = [A.Resize(height=image_size, width=image_size)]
        if split == "train":
            transforms.append(A.HorizontalFlip(p=0.5))
        transforms.extend([A.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD), ToTensorV2()])
        self.transform = A.Compose(
            transforms,
            bbox_params=A.BboxParams(
                format="pascal_voc",
                label_fields=["labels", "iscrowd"],
                clip=True,
                filter_invalid_bboxes=True,
            ),
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, dict[str, Tensor]]:
        record = self.records[index]
        try:
            with Image.open(record.image_path) as source_image:
                image = np.asarray(source_image.convert("RGB")).copy()
        except (OSError, ValueError) as error:
            raise OSError(f"could not load image {record.image_path}: {error}") from error
        height, width = image.shape[:2]
        paths = _find_annotation_paths(
            record,
            self._split_roots[record.source_split],
            self._flat_indexes.get(record.source_split, {}),
        )
        boxes, labels, ignore_boxes = [], [], []
        for path in paths:
            object_boxes, object_labels, ignored = _annotation_boxes(path, width, height)
            boxes.extend(object_boxes)
            labels.extend(object_labels)
            ignore_boxes.extend(ignored)
        if self.mask_ignore_pixels and ignore_boxes:
            mean_rgb = np.rint(image.astype(np.float32).mean(axis=(0, 1))).clip(0, 255).astype(np.uint8)
            for x1, y1, x2, y2 in ignore_boxes:
                image[int(math.floor(y1)):int(math.ceil(y2)), int(math.floor(x1)):int(math.ceil(x2))] = mean_rgb
        transformed = self.transform(
            image=image,
            bboxes=boxes + ignore_boxes,
            labels=labels + [1] * len(ignore_boxes),
            iscrowd=[0] * len(boxes) + [1] * len(ignore_boxes),
        )
        target = {
            "boxes": torch.as_tensor(transformed["bboxes"], dtype=torch.float32).reshape(-1, 4),
            "labels": torch.as_tensor(transformed["labels"], dtype=torch.int64),
            "iscrowd": torch.as_tensor(transformed["iscrowd"], dtype=torch.int64),
            "image_id": torch.tensor(index, dtype=torch.int64),
        }
        return transformed["image"], target


# Stale split manifests are removed above; load a valid manifest when present,
# otherwise rebuild it from the dataset's train/val folders.
EDA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = Path(SHARED_CONFIG["manifest"])

try:
    if not MANIFEST_PATH.is_file():
        raise FileNotFoundError(f"manifest not found: {MANIFEST_PATH}")
    RECORDS, DATA_SOURCE = load_records(MANIFEST_PATH, DATA_ROOT)
except (OSError, ValueError) as manifest_error:
    LOGGER.warning("Manifest unavailable or invalid (%s); falling back to split scanning.", manifest_error)
    try:
        raw_records = _scan_split(DATA_ROOT, "train") + _scan_split(DATA_ROOT, "val")
    except (OSError, ValueError) as scan_error:
        raise RuntimeError(
            f"Could not load the manifest ({manifest_error}) or scan train/val folders "
            f"under {DATA_ROOT} ({scan_error}). Verify the Kaggle download and DATA_ROOT."
        ) from scan_error

    manifest_rows = [
        {
            "clip_id": f"{record.source_split}:{record.clip_id}",
            "assigned_split": record.source_split,
            "source_split": record.source_split,
            "frame_id": record.frame_id,
            "image_path": str(record.image_path),
        }
        for record in raw_records
    ]
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(manifest_rows).to_csv(MANIFEST_PATH, index=False)
    RECORDS, DATA_SOURCE = load_records(MANIFEST_PATH, DATA_ROOT)


FRAME_SKIP = int(SHARED_CONFIG["frame_skip"])
TRAIN_RECORDS = [record for record in RECORDS if record.assigned_split == "train"][::FRAME_SKIP]
VAL_RECORDS = [record for record in RECORDS if record.assigned_split == "val"][::FRAME_SKIP]
if not TRAIN_RECORDS or not VAL_RECORDS:
    raise ValueError("Manifest and frame_skip must leave non-empty train and validation sets.")

TRAIN_DATASET = VisDroneDetectionDataset(TRAIN_RECORDS, DATA_ROOT, "train", SHARED_CONFIG["image_size"])
VAL_DATASET = VisDroneDetectionDataset(VAL_RECORDS, DATA_ROOT, "val", SHARED_CONFIG["image_size"])
LOGGER.info("Data source=%s | train=%d | val=%d", DATA_SOURCE, len(TRAIN_DATASET), len(VAL_DATASET))


def prepare_rtdetr_dataset() -> dict[str, Path]:
    destination = OUTPUT_DIR / "rtdetr_submission_data"
    image_dirs = {split: destination / "images" / split for split in ("train", "val")}
    label_dirs = {split: destination / "labels" / split for split in ("train", "val")}
    for directory in (*image_dirs.values(), *label_dirs.values()):
        directory.mkdir(parents=True, exist_ok=True)

    source_roots = {
        split: (SPLIT_ROOTS.get(split) or (DATA_ROOT / split)).resolve()
        for split in {record.source_split for record in TRAIN_RECORDS + VAL_RECORDS}
    }
    flat_indexes = {
        split: _flat_annotation_paths(root)
        for split, root in source_roots.items()
        if not (root / "labels").is_dir() and not (root / "ignore").is_dir()
    }
    list_paths: dict[str, Path] = {}
    for split, split_records in (("train", TRAIN_RECORDS), ("val", VAL_RECORDS)):
        image_list = []
        for index, record in enumerate(split_records):
            with Image.open(record.image_path) as source_image:
                image = np.asarray(source_image.convert("RGB")).copy()
            height, width = image.shape[:2]
            annotation_paths = _find_annotation_paths(
                record, source_roots[record.source_split], flat_indexes.get(record.source_split, {})
            )
            boxes, labels, ignores = [], [], []
            for annotation_path in annotation_paths:
                object_boxes, object_labels, ignore_boxes = _annotation_boxes(annotation_path, width, height)
                boxes.extend(object_boxes)
                labels.extend(object_labels)
                ignores.extend(ignore_boxes)
            if ignores:
                mean_rgb = np.rint(image.astype(np.float32).mean(axis=(0, 1))).clip(0, 255).astype(np.uint8)
                for x1, y1, x2, y2 in ignores:
                    image[int(math.floor(y1)):int(math.ceil(y2)), int(math.floor(x1)):int(math.ceil(x2))] = mean_rgb

            filename = f"{index:08d}{record.image_path.suffix.lower()}"
            image_path = image_dirs[split] / filename
            label_path = label_dirs[split] / f"{Path(filename).stem}.txt"
            Image.fromarray(image).resize(
                (SHARED_CONFIG["image_size"], SHARED_CONFIG["image_size"])
            ).save(image_path)
            label_lines = []
            for box, one_based_label in zip(boxes, labels):
                class_id = one_based_label - 1
                if not 0 <= class_id < SHARED_CONFIG["num_classes"]:
                    raise ValueError(f"invalid VisDrone class id: {class_id}")
                x1, y1, x2, y2 = box
                label_lines.append(
                    f"{class_id} {((x1 + x2) / 2) / width:.6f} "
                    f"{((y1 + y2) / 2) / height:.6f} "
                    f"{(x2 - x1) / width:.6f} {(y2 - y1) / height:.6f}"
                )
            label_path.write_text("\n".join(label_lines), encoding="utf-8")
            image_list.append(str(image_path.resolve()))
        list_path = destination / f"{split}.txt"
        list_path.write_text("\n".join(image_list) + "\n", encoding="utf-8")
        list_paths[split] = list_path

    data_yaml = destination / "data.yaml"
    data_yaml.write_text(
        f"path: {json.dumps(str(destination.resolve()))}\n"
        f"train: {json.dumps(str(list_paths['train'].resolve()))}\n"
        f"val: {json.dumps(str(list_paths['val'].resolve()))}\n"
        "names:\n"
        + "".join(f"  {index}: {name}\n" for index, name in enumerate(CLASS_NAMES)),
        encoding="utf-8",
    )
    return {"data_yaml": data_yaml, "dataset_dir": destination}


RTDETR_DATASET = prepare_rtdetr_dataset()
LOGGER.info("Generated RT-DETR data.yaml: %s", RTDETR_DATASET["data_yaml"])


## Cell 4: Train the 800px RT-DETR Model


In [5]:
SUBMISSION_CHECKPOINT = CHECKPOINT_DIR / "rtdetr_best_800px.pt"

def set_rtdetr_num_queries(model: RTDETR, num_queries: int) -> None:
    if num_queries < 1:
        raise ValueError("num_queries must be positive")
    decoders = [module for module in model.model.modules() if module.__class__.__name__ == "RTDETRDecoder"]
    if len(decoders) != 1:
        raise RuntimeError(f"expected one RTDETRDecoder, found {len(decoders)}")
    decoders[0].num_queries = int(num_queries)


torch.manual_seed(TRAIN_CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAIN_CONFIG["seed"])

model = RTDETR(RTDETR_EXPERIMENT_CONFIG["model"])
set_rtdetr_num_queries(model, 500)

def configure_matching_costs(trainer) -> None:
    from ultralytics.models.utils.loss import RTDETRDetectionLoss

    criterion = RTDETRDetectionLoss(nc=trainer.model.nc, use_vfl=True)
    criterion.matcher.cost_gain = {
        "class": RTDETR_EXPERIMENT_CONFIG["matching_cost_class"],
        "bbox": RTDETR_EXPERIMENT_CONFIG["matching_cost_bbox"],
        "giou": RTDETR_EXPERIMENT_CONFIG["matching_cost_giou"],
    }
    trainer.model.criterion = criterion
    LOGGER.info("RT-DETR matcher costs: %s", criterion.matcher.cost_gain)

model.add_callback("on_train_start", configure_matching_costs)
model.train(
    data=str(RTDETR_DATASET["data_yaml"]),
    epochs=12,
    imgsz=800,
    batch=TRAIN_CONFIG["batch"],
    device=TRAIN_CONFIG["device"],
    project=str(OUTPUT_DIR),
    name="rtdetr_best_pipeline_800px",
    seed=TRAIN_CONFIG["seed"],
    pretrained=True,
    max_det=500,
    cos_lr=True,
    amp=True,
    augment=True,
    workers=2,
    plots=False,
    verbose=True,
)

best_path = Path(model.trainer.save_dir) / "weights" / "best.pt"
if not best_path.is_file():
    best_path = Path(model.trainer.save_dir) / "weights" / "last.pt"
if not best_path.is_file():
    raise FileNotFoundError("Ultralytics did not produce best.pt or last.pt.")
model = RTDETR(str(best_path))
model.save(str(SUBMISSION_CHECKPOINT))
if not SUBMISSION_CHECKPOINT.is_file():
    raise FileNotFoundError(f"could not save high-resolution checkpoint: {SUBMISSION_CHECKPOINT}")
LOGGER.info("Saved high-resolution checkpoint: %s", SUBMISSION_CHECKPOINT)


Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/outputs/rtdetr_submission_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=12, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=500, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_best_pipeline_800px, nbs=64, nms=

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/12      8.97G      1.173     0.5054     0.2635         23        800: 100% ━━━━━━━━━━━━ 606/606 1.4it/s 7:280.8ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 1.9it/s 18.5s0.5s
                   all        570      22899      0.389      0.469      0.371      0.191

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/12       9.1G     0.7594     0.5272     0.1064        233        800: 0% ──────────── 0/606  0.7s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/12      9.18G     0.8064     0.4597    0.09915         30        800: 100% ━━━━━━━━━━━━ 606/606 1.4it/s 7:130.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 2.2it/s 16.6s0.5s
                   all        570      22899      0.491      0.437      0.413      0.219
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/12      9.17G     0.7884     0.4414    0.09166        537        800: 0% ──────────── 0/606  1.5s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (15 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      11/12      8.94G     0.5209     0.4157     0.0497         55        800: 100% ━━━━━━━━━━━━ 606/606 1.4it/s 7:000.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 2.2it/s 16.5s0.5s
                   all        570      22899      0.548      0.409      0.403      0.208

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      12/12      9.12G     0.4772     0.4055    0.03553        188        800: 0% ──────────── 0/606  0.7s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      12/12      9.16G     0.5175     0.4132    0.04915         31        800: 100% ━━━━━━━━━━━━ 606/606 1.4it/s 7:000.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 2.2it/s 16.7s0.5s
                   all        570      22899       0.51      0.432      0.399      0.206

12 epochs completed in 1.540 hours.
Optimizer stripped from /content/outputs/rtdetr_best_pipeline_800px/weights/last.pt, 66.3MB
Optimizer stripped from /content/outputs/rtdetr_best_pipeline_800px/weights/best.pt, 66.3MB

Validating /content/outputs/rtdetr_best_pipeline_800px/weights/best.pt...
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 2.3it/s 16.0s0.4s
                   all        570      22899      0.587 

# Cell 4.5 post training optimisation

In [6]:
import shutil
from pathlib import Path
from ultralytics import RTDETR

data_yaml = '/content/outputs/rtdetr_submission_data/data.yaml'
best_weights = Path('/content/outputs/rtdetr_best_pipeline_800px/weights/best.pt')
last_weights = Path('/content/outputs/rtdetr_best_pipeline_800px/weights/last.pt')
submission_checkpoint = Path('/content/outputs/checkpoints/rtdetr_best_800px.pt')

print("=== Step 1: Evaluating best.pt vs last.pt ===")
model_best = RTDETR(str(best_weights))
res_best = model_best.val(data=data_yaml, imgsz=800, split='val', verbose=False)
print(f"best.pt  -> mAP50-95: {res_best.box.map:.4f} | mAP50: {res_best.box.map50:.4f}")

if last_weights.is_file():
    model_last = RTDETR(str(last_weights))
    res_last = model_last.val(data=data_yaml, imgsz=800, split='val', verbose=False)
    print(f"last.pt  -> mAP50-95: {res_last.box.map:.4f} | mAP50: {res_last.box.map50:.4f}")

print("\n=== Step 2: Running Confidence Threshold Sweep on best.pt ===")
best_conf = 0.25
best_map = res_best.box.map

for conf_val in [0.05, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25]:
    metrics = model_best.val(data=data_yaml, imgsz=800, conf=conf_val, split='val', verbose=False)
    map_score = metrics.box.map
    map50_score = metrics.box.map50
    print(f"Conf: {conf_val:.2f} | mAP50-95: {map_score:.4f} | mAP50: {map50_score:.4f}")
    
    if map_score > best_map:
        best_map = map_score
        best_conf = conf_val

# Lock in the winning checkpoint and global threshold
submission_checkpoint.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best_weights, submission_checkpoint)

print(f"\nSaved winning checkpoint to: {submission_checkpoint}")
print(f"Optimal Inference Threshold locked: conf={best_conf:.2f} (Validation mAP50-95: {best_map:.4f})")

=== Step 1: Evaluating best.pt vs last.pt ===
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2024.6±740.5 MB/s, size: 109.7 KB)
val: Scanning /content/outputs/rtdetr_submission_data/labels/val.cache... 570 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 570/570 183.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 1.0it/s 35.8s1.0ss
                   all        570      22899      0.587      0.458      0.445      0.229
Speed: 1.7ms preprocess, 55.8ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val
best.pt  -> mAP50-95: 0.2295 | mAP50: 0.4453
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gra

## Cell 5: Test Inference & Competition Prediction Strings


In [7]:
# Load the verified high-resolution checkpoint and optimal threshold
if 'best_conf' not in globals():
    best_conf = SHARED_CONFIG.get("confidence_threshold", 0.12)

model = RTDETR(str(SUBMISSION_CHECKPOINT))

TEST_CSV_CANDIDATES = (
    DATA_ROOT / "test_images.csv",
    OUTPUT_DIR / "eda" / "test_images.csv",
)
TEST_CSV = next((path for path in TEST_CSV_CANDIDATES if path.is_file()), None)
if TEST_CSV is None:
    raise FileNotFoundError(
        "Could not find test_images.csv at DATA_ROOT/test_images.csv or OUTPUT_DIR/eda/test_images.csv."
    )

test_images_df = pd.read_csv(TEST_CSV, dtype={"image_id": str, "file_name": str})
required_columns = {"image_id", "file_name", "width", "height"}
if not required_columns.issubset(test_images_df.columns):
    raise ValueError(f"test_images.csv must contain columns: {sorted(required_columns)}")
if test_images_df["image_id"].isna().any() or test_images_df["file_name"].isna().any():
    raise ValueError("test_images.csv contains missing image_id or file_name values.")


def resolve_test_image(file_name: str) -> Path:
    relative_path = Path(file_name).expanduser()
    candidates = [relative_path] if relative_path.is_absolute() else [
        DATA_ROOT / relative_path,
        DATA_ROOT / "test" / relative_path,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"test image does not exist: {file_name}")


prediction_rows = []
inference_device = 0 if torch.cuda.is_available() else DEVICE
for row_index, row in enumerate(test_images_df.itertuples(index=False), start=1):
    image_path = resolve_test_image(row.file_name)
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise OSError(f"could not read test image: {image_path}")
    image_height, image_width = image_bgr.shape[:2]
    result = model.predict(
    source=image_bgr,
    imgsz=800,
    conf=best_conf,  # <--- Use optimal threshold from Cell 4.5
    max_det=500,
    device=inference_device,
    verbose=False,
    )[0]

    tokens = []
    if result.boxes is not None and len(result.boxes):
        boxes = result.boxes.xyxy.detach().cpu().numpy()
        scores = result.boxes.conf.detach().cpu().numpy()
        class_ids = result.boxes.cls.detach().cpu().numpy().astype(np.int64)
        order = np.argsort(-scores)[:500]
        for detection_index in order:
            class_id = int(class_ids[detection_index])
            if not 0 <= class_id < SHARED_CONFIG["num_classes"]:
                raise ValueError(f"prediction class id outside 0..7: {class_id}")
            x1, y1, x2, y2 = boxes[detection_index].astype(float)
            x1 = float(np.clip(x1, 0, image_width))
            y1 = float(np.clip(y1, 0, image_height))
            x2 = float(np.clip(x2, 0, image_width))
            y2 = float(np.clip(y2, 0, image_height))
            width, height = x2 - x1, y2 - y1
            if width <= 0 or height <= 0:
                continue
            tokens.extend(
                (
                    str(class_id),
                    f"{float(scores[detection_index]):.6f}",
                    f"{x1:.2f}",
                    f"{y1:.2f}",
                    f"{width:.2f}",
                    f"{height:.2f}",
                )
            )
    prediction_rows.append(
        {"image_id": str(row.image_id), "PredictionString": " ".join(tokens)}
    )
    if row_index % 250 == 0 or row_index == len(test_images_df):
        LOGGER.info("Inferred %d/%d test frames", row_index, len(test_images_df))


## Cell 6: Submission Verification & CSV Export


In [8]:
submission_df = pd.DataFrame(prediction_rows, columns=["image_id", "PredictionString"])

EXPECTED_TEST_FRAMES = 3811
assert len(submission_df) == len(test_images_df), (
    f"submission rows ({len(submission_df)}) do not match test rows ({len(test_images_df)})"
)
assert len(submission_df) == EXPECTED_TEST_FRAMES, (
    f"expected {EXPECTED_TEST_FRAMES} test frames, found {len(submission_df)}"
)
assert submission_df.columns.tolist() == ["image_id", "PredictionString"]
assert not submission_df["image_id"].isna().any()
assert submission_df["image_id"].is_unique
assert not submission_df["PredictionString"].isna().any()

submission_path = SUBMISSION_DIR / "submission.csv"
submission_df.to_csv(submission_path, index=False)
verified_df = pd.read_csv(
    submission_path,
    dtype={"image_id": str},
    keep_default_na=False,
)
assert len(verified_df) == EXPECTED_TEST_FRAMES
assert verified_df.columns.tolist() == ["image_id", "PredictionString"]
assert not verified_df.isna().any().any()

detection_counts = verified_df["PredictionString"].map(
    lambda value: len(value.split()) // 6 if value else 0
)
assert (detection_counts <= SHARED_CONFIG["max_detections"]).all()
print(f"Saved valid submission: {submission_path}")
print(verified_df.head(5).to_string(index=False))
print(
    "Summary: "
    f"frames={len(verified_df)}, "
    f"non-empty_frames={(detection_counts > 0).sum()}, "
    f"total_detections={int(detection_counts.sum())}, "
    f"average_detections_per_frame={detection_counts.mean():.3f}"
)


Saved valid submission: /content/outputs/submissions/submission.csv
        image_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    